In [16]:
import os

# Create Medallion architecture directories
for folder in ["bronze", "silver", "gold"]:
    os.makedirs(folder, exist_ok=True)

# Download and unzip the Book-Crossing dataset
!wget -q https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
!unzip -o -q book-crossings.zip -d raw_data/
!ls -lh raw_data/

total 116M
-rw-rw-rw- 1 root root 30M Oct 11  2004 BX-Book-Ratings.csv
-rw-rw-rw- 1 root root 75M Oct 11  2004 BX-Books.csv
-rw-rw-rw- 1 root root 12M Oct 11  2004 BX-Users.csv


In [17]:
import pandas as pd

ingestion_time = pd.Timestamp.now()

# Load raw CSVs
users_raw = pd.read_csv(
    "raw_data/BX-Users.csv",
    sep=";",
    encoding="latin-1",
    on_bad_lines="skip",
    low_memory=False
)

books_raw = pd.read_csv(
    "raw_data/BX-Books.csv",
    sep=";",
    encoding="latin-1",
    escapechar="\\",
    on_bad_lines="skip",
    low_memory=False
)

ratings_raw = pd.read_csv(
    "raw_data/BX-Book-Ratings.csv",
    sep=";",
    encoding="latin-1",
    on_bad_lines="skip",
    low_memory=False
)

# Add ingestion timestamp without cleaning any data
for df in (users_raw, books_raw, ratings_raw):
    df["ingestion_timestamp"] = ingestion_time

# Ensure mixed-type object columns don't break Parquet serialization
books_raw["Year-Of-Publication"] = books_raw["Year-Of-Publication"].astype(str)

# Save to bronze/ as Parquet
users_raw.to_parquet("bronze/users.parquet", index=False)
books_raw.to_parquet("bronze/books.parquet", index=False)
ratings_raw.to_parquet("bronze/ratings.parquet", index=False)

print("Bronze layer saved successfully:")
print(f"Users:   {users_raw.shape}")
print(f"Books:   {books_raw.shape}")
print(f"Ratings: {ratings_raw.shape}")

Bronze layer saved successfully:
Users:   (278858, 4)
Books:   (271379, 9)
Ratings: (1149780, 4)


In [18]:
import pandas as pd
import numpy as np

# Read back from the Bronze layer
users_b = pd.read_parquet("bronze/users.parquet")
books_b = pd.read_parquet("bronze/books.parquet")
ratings_b = pd.read_parquet("bronze/ratings.parquet")

for name, df in [("Users", users_b), ("Books", books_b), ("Ratings", ratings_b)]:
    print(f"=== {name} ===")
    print("Shape:", df.shape)
    summary = pd.DataFrame({
        "dtype": df.dtypes,
        "missing_count": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(2)
    })
    print(summary)
    print()

=== Users ===
Shape: (278858, 4)
                              dtype  missing_count  missing_pct
User-ID                       int64              0         0.00
Location                     object              0         0.00
Age                         float64         110762        39.72
ingestion_timestamp  datetime64[us]              0         0.00

=== Books ===
Shape: (271379, 9)
                              dtype  missing_count  missing_pct
ISBN                         object              0          0.0
Book-Title                   object              0          0.0
Book-Author                  object              2          0.0
Year-Of-Publication          object              0          0.0
Publisher                    object              2          0.0
Image-URL-S                  object              0          0.0
Image-URL-M                  object              0          0.0
Image-URL-L                  object              0          0.0
ingestion_timestamp  datetime64[us]  

**Finding & Silver Plan:** `Users` has 110,762 (39.72%) missing `Age` values, `Books` has 2 missing authors/publishers and stores `Year-Of-Publication` as a string, and `Ratings` has no nulls; in Silver, we will cast publication year to numeric, impute ages with the median, and drop unused image-URL columns.

In [19]:
rating_counts = ratings_b["Book-Rating"].value_counts().sort_index()
zero_count = (ratings_b["Book-Rating"] == 0).sum()
zero_pct = (zero_count / len(ratings_b)) * 100

print("Rating value distribution:")
print(rating_counts)
print(f"\nRatings equal to 0: {zero_count:,} ({zero_pct:.2f}% of all ratings)")

Rating value distribution:
Book-Rating
0     716109
1       1770
2       2759
3       5996
4       8904
5      50974
6      36924
7      76457
8     103736
9      67541
10     78610
Name: count, dtype: int64

Ratings equal to 0: 716,109 (62.28% of all ratings)


**Finding & Silver Plan:** 716,109 ratings (62.28%) are `0` (implicit/no explicit rating), while the remaining 433,671 are explicit ratings from 1 to 10; in Silver, we will drop all `0` ratings and keep only explicit ratings (`1–10`).

In [20]:
# 1. User Ages
print("=== User Age Distribution ===")
print(users_b["Age"].describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))
print("Ages < 5:", (users_b["Age"] < 5).sum())
print("Ages > 100:", (users_b["Age"] > 100).sum())
print("Null ages:", users_b["Age"].isna().sum())

# 2. Publication Years
print("\n=== Publication Year Distribution ===")
# Convert to numeric temporarily for EDA (errors='coerce' catches non-numeric strings)
pub_years = pd.to_numeric(books_b["Year-Of-Publication"], errors="coerce")
print(pub_years.describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99]))
print("Non-numeric years:", pub_years.isna().sum())
print("Year == 0:", (pub_years == 0).sum())
print("Future years (> 2026):", (pub_years > 2026).sum())
print("Sample weird years:", books_b.loc[pub_years.isna() | (pub_years == 0) | (pub_years > 2026), "Year-Of-Publication"].unique()[:15])

=== User Age Distribution ===
count    168096.000000
mean         34.751434
std          14.428097
min           0.000000
1%           13.000000
5%           16.000000
50%          32.000000
95%          60.000000
99%          71.000000
max         244.000000
Name: Age, dtype: float64
Ages < 5: 882
Ages > 100: 366
Null ages: 110762

=== Publication Year Distribution ===
count    271379.000000
mean       1959.756050
std         258.011363
min           0.000000
1%            0.000000
5%         1976.000000
50%        1995.000000
95%        2003.000000
99%        2004.000000
max        2050.000000
Name: Year-Of-Publication, dtype: float64
Non-numeric years: 0
Year == 0: 4619
Future years (> 2026): 11
Sample weird years: ['0' '2030' '2050' '2038' '2037']


**Finding & Silver Plan:** `Users` contains impossible ages (882 below 5 and 366 above 100, up to 244) and `Books` has 4,619 zero years and 11 future years (up to 2050); in Silver, we will set out-of-range ages and invalid/future years to null and fill missing ages with the median.

In [21]:
user_rating_counts = ratings_b.groupby("User-ID")["Book-Rating"].count()
book_rating_counts = ratings_b.groupby("ISBN")["Book-Rating"].count()

print("=== Ratings per User ===")
print(user_rating_counts.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))
print(f"Users with < 10 ratings: {(user_rating_counts < 10).sum():,} ({(user_rating_counts < 10).mean()*100:.2f}%)")

print("\n=== Ratings per Book ===")
print(book_rating_counts.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))
print(f"Books with < 10 ratings: {(book_rating_counts < 10).sum():,} ({(book_rating_counts < 10).mean()*100:.2f}%)")

=== Ratings per User ===
count    105283.000000
mean         10.920851
std          90.562825
min           1.000000
50%           1.000000
75%           4.000000
90%          12.000000
95%          29.000000
99%         174.000000
max       13602.000000
Name: Book-Rating, dtype: float64
Users with < 10 ratings: 92,186 (87.56%)

=== Ratings per Book ===
count    340556.000000
mean          3.376185
std          12.436252
min           1.000000
50%           1.000000
75%           2.000000
90%           6.000000
95%          10.000000
99%          36.000000
max        2502.000000
Name: Book-Rating, dtype: float64
Books with < 10 ratings: 322,237 (94.62%)


**Finding & Silver Plan:** Rating activity is heavily skewed, 87.56% of users and 94.62% of books have fewer than 10 ratings (median is just 1); in Silver, we will filter to users and books with at least 10 explicit ratings so collaborative filtering has reliable signal.

In [22]:
n_users = ratings_b["User-ID"].nunique()
n_books = ratings_b["ISBN"].nunique()
n_ratings = len(ratings_b)

possible_pairs = n_users * n_books
filled_pct = (n_ratings / possible_pairs) * 100
sparsity_pct = 100 - filled_pct

print(f"Unique users in ratings: {n_users:,}")
print(f"Unique books in ratings: {n_books:,}")
print(f"Total ratings:           {n_ratings:,}")
print(f"Possible user-book pairs: {possible_pairs:,}")
print(f"Filled percentage (density): {filled_pct:.6f}%")
print(f"Empty percentage (sparsity): {sparsity_pct:.6f}%")

Unique users in ratings: 105,283
Unique books in ratings: 340,556
Total ratings:           1,149,780
Possible user-book pairs: 35,854,757,348
Filled percentage (density): 0.003207%
Empty percentage (sparsity): 99.996793%


**Finding & Silver Plan:** Only 0.003207% of possible user-book pairs have a rating (99.996793% sparsity); in Silver, we will remove low-activity users and books to reduce sparsity, and in Gold, we will store the rating matrix as a sparse matrix.

In [23]:
import pandas as pd
import numpy as np

# Load from Bronze
users_s = pd.read_parquet("bronze/users.parquet")
books_s = pd.read_parquet("bronze/books.parquet")
ratings_s = pd.read_parquet("bronze/ratings.parquet")

# 1. Remove duplicate rows
users_s = users_s.drop_duplicates().drop_duplicates(subset=["User-ID"])
books_s = books_s.drop_duplicates().drop_duplicates(subset=["ISBN"])
ratings_s = ratings_s.drop_duplicates()

# 2. Remove ratings of 0 and keep explicit ratings (1 to 10)
ratings_s = ratings_s[ratings_s["Book-Rating"].between(1, 10)].copy()

# 3. Fix user ages: set < 5 or > 100 to null, then fill nulls with median age
users_s.loc[(users_s["Age"] < 5) | (users_s["Age"] > 100), "Age"] = np.nan
median_age = users_s["Age"].median()
users_s["Age"] = users_s["Age"].fillna(median_age)

# 4. Split Location into city, state, country
loc_parts = users_s["Location"].astype(str).str.split(",", n=2, expand=True)
users_s["city"] = loc_parts[0].str.strip().str.lower()
users_s["state"] = loc_parts[1].str.strip().str.lower() if loc_parts.shape[1] > 1 else np.nan
users_s["country"] = loc_parts[2].str.strip().str.lower() if loc_parts.shape[1] > 2 else np.nan

# 5. Fix publication years: set 0 or future years (> 2026) to null
books_s["Year-Of-Publication"] = pd.to_numeric(books_s["Year-Of-Publication"], errors="coerce")
books_s.loc[
    (books_s["Year-Of-Publication"] == 0) | (books_s["Year-Of-Publication"] > 2026),
    "Year-Of-Publication"
] = np.nan

# 6. Clean text columns (strip extra spaces, standardise case) & drop image-URL columns
books_s = books_s.drop(columns=["Image-URL-S", "Image-URL-M", "Image-URL-L"], errors="ignore")

for col in ["Book-Title", "Book-Author", "Publisher"]:
    books_s[col] = books_s[col].astype(str).str.strip().str.replace(r"\s+", " ", regex=True).str.lower()

# Clean ISBN whitespace so joins match cleanly
books_s["ISBN"] = books_s["ISBN"].astype(str).str.strip()
ratings_s["ISBN"] = ratings_s["ISBN"].astype(str).str.strip()

# 7. Referential integrity: keep only ratings whose book and user exist in books and users
ratings_s = ratings_s[
    ratings_s["ISBN"].isin(books_s["ISBN"]) &
    ratings_s["User-ID"].isin(users_s["User-ID"])
].copy()

# 8. Filter to users and books with at least MIN_RATINGS explicit ratings
MIN_RATINGS = 10

user_counts = ratings_s["User-ID"].value_counts()
active_users = user_counts[user_counts >= MIN_RATINGS].index

book_counts = ratings_s["ISBN"].value_counts()
popular_books = book_counts[book_counts >= MIN_RATINGS].index

ratings_s = ratings_s[
    ratings_s["User-ID"].isin(active_users) &
    ratings_s["ISBN"].isin(popular_books)
].copy()

# Filter users and books tables to match the remaining ratings
users_s = users_s[users_s["User-ID"].isin(ratings_s["User-ID"])].copy()
books_s = books_s[books_s["ISBN"].isin(ratings_s["ISBN"])].copy()

# 9. Save cleaned tables to silver/
users_s.to_parquet("silver/users.parquet", index=False)
books_s.to_parquet("silver/books.parquet", index=False)
ratings_s.to_parquet("silver/ratings.parquet", index=False)

# Print summary to verify
n_u = ratings_s["User-ID"].nunique()
n_b = ratings_s["ISBN"].nunique()
new_density = (len(ratings_s) / (n_u * n_b)) * 100

print("Silver layer saved successfully:")
print(f"Median age used for imputation: {median_age}")
print(f"Users:   {users_s.shape}")
print(f"Books:   {books_s.shape}")
print(f"Ratings: {ratings_s.shape}")
print(f"New matrix density: {new_density:.4f}% (sparsity: {100 - new_density:.4f}%)")

Silver layer saved successfully:
Median age used for imputation: 32.0
Users:   (6318, 7)
Books:   (5438, 6)
Ratings: (87178, 4)
New matrix density: 0.2537% (sparsity: 99.7463%)


**Threshold Justification:** We chose a minimum threshold of **10 explicit ratings** per user and per book. From our Bronze EDA, 87.56% of users and 94.62% of books had fewer than 10 ratings (with a median of 1), resulting in 99.9968% sparsity. Filtering at >= 10 removes cold-start noise while retaining enough active users and popular books for meaningful item-based cosine similarity.

In [24]:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix, save_npz

# Load cleaned tables from Silver
users_g = pd.read_parquet("silver/users.parquet")
books_g = pd.read_parquet("silver/books.parquet")
ratings_g = pd.read_parquet("silver/ratings.parquet")

# 1. Create User-Book Sparse Rating Matrix
# Map User-ID and ISBN to contiguous integer indices
user_ids = ratings_g["User-ID"].unique()
book_isbns = ratings_g["ISBN"].unique()

user_to_idx = {uid: idx for idx, uid in enumerate(user_ids)}
book_to_idx = {isbn: idx for idx, isbn in enumerate(book_isbns)}

row_idx = ratings_g["User-ID"].map(user_to_idx).values
col_idx = ratings_g["ISBN"].map(book_to_idx).values
rating_vals = ratings_g["Book-Rating"].astype(np.float32).values

user_book_matrix = csr_matrix(
    (rating_vals, (row_idx, col_idx)),
    shape=(len(user_ids), len(book_isbns))
)

# Save sparse matrix and user index mapping to gold/
save_npz("gold/user_book_matrix.npz", user_book_matrix)

user_lookup = pd.DataFrame({
    "user_idx": range(len(user_ids)),
    "User-ID": user_ids
})
user_lookup.to_parquet("gold/user_lookup.parquet", index=False)

# 2. Create Book Lookup Table (book id, title, author + matrix index)
book_lookup = pd.DataFrame({
    "book_idx": range(len(book_isbns)),
    "ISBN": book_isbns
}).merge(
    books_g[["ISBN", "Book-Title", "Book-Author"]],
    on="ISBN",
    how="left"
)
book_lookup.to_parquet("gold/book_lookup.parquet", index=False)

# 3. User-level and Book-level Aggregates (average rating and number of ratings)
user_aggregates = (
    ratings_g.groupby("User-ID", as_index=False)["Book-Rating"]
    .agg(avg_rating="mean", rating_count="count")
)
user_aggregates["avg_rating"] = user_aggregates["avg_rating"].round(2)
user_aggregates.to_parquet("gold/user_aggregates.parquet", index=False)

book_aggregates = (
    ratings_g.groupby("ISBN", as_index=False)["Book-Rating"]
    .agg(avg_rating="mean", rating_count="count")
    .merge(books_g[["ISBN", "Book-Title", "Book-Author"]], on="ISBN", how="left")
)
book_aggregates["avg_rating"] = book_aggregates["avg_rating"].round(2)
book_aggregates.to_parquet("gold/book_aggregates.parquet", index=False)

print("Gold layer saved successfully:")
print(f"Sparse user-book matrix shape: {user_book_matrix.shape} | Non-zero entries: {user_book_matrix.nnz:,}")
print(f"Book lookup table:   {book_lookup.shape}")
print(f"User aggregates:     {user_aggregates.shape}")
print(f"Book aggregates:     {book_aggregates.shape}")
print("\nSample Book Lookup:")
print(book_lookup.head(3))
print("\nTop 5 Most-Rated Books:")
print(book_aggregates.sort_values("rating_count", ascending=False).head(5)[["Book-Title", "Book-Author", "rating_count", "avg_rating"]])

Gold layer saved successfully:
Sparse user-book matrix shape: (6318, 5438) | Non-zero entries: 87,178
Book lookup table:   (5438, 4)
User aggregates:     (6318, 3)
Book aggregates:     (5438, 5)

Sample Book Lookup:
   book_idx        ISBN                           Book-Title   Book-Author
0         0  0060096195                    the boy next door  meggin cabot
1         1  0375821813            hoot (newbery honor book)  carl hiaasen
2         2  0786817070  artemis fowl (artemis fowl, book 1)   eoin colfer

Top 5 Most-Rated Books:
                                             Book-Title    Book-Author  \
848                           the lovely bones: a novel   alice sebold   
1774                                  the da vinci code      dan brown   
688                 the red tent (bestselling backlist)  anita diamant   
3791  harry potter and the sorcerer's stone (harry p...  j. k. rowling   
5084                                        wild animus   rich shapero   

      rating_c

In [25]:
import pandas as pd
import numpy as np
from scipy.sparse import load_npz
from sklearn.neighbors import NearestNeighbors

# 1. Load from Gold layer and arrange so rows = books, columns = users
user_book_matrix = load_npz("gold/user_book_matrix.npz")
book_user_matrix = user_book_matrix.T.tocsr()

book_lookup = pd.read_parquet("gold/book_lookup.parquet")
user_lookup = pd.read_parquet("gold/user_lookup.parquet")

# Fast lookup dictionaries
user_id_to_idx = dict(zip(user_lookup["User-ID"], user_lookup["user_idx"]))
book_idx_to_info = book_lookup.set_index("book_idx")[["ISBN", "Book-Title", "Book-Author"]].to_dict("index")

print(f"Loaded book-user matrix: {book_user_matrix.shape} (rows=books, cols=users)")

# 2. Fit NearestNeighbors with cosine metric on the sparse matrix
nn_model = NearestNeighbors(metric="cosine", algorithm="brute")
nn_model.fit(book_user_matrix)

# 3. Function: given a book title, return the 10 most similar books
def get_similar_books(book_title, n=10):
    query = book_title.strip().lower()

    # Exact match first, fallback to substring match
    matches = book_lookup[book_lookup["Book-Title"] == query]
    if matches.empty:
        matches = book_lookup[book_lookup["Book-Title"].str.contains(query, regex=False, na=False)]
    if matches.empty:
        print(f"Book '{book_title}' not found in Gold layer.")
        return pd.DataFrame()

    target_idx = matches.iloc[0]["book_idx"]
    matched_title = matches.iloc[0]["Book-Title"]

    distances, indices = nn_model.kneighbors(book_user_matrix[target_idx], n_neighbors=n + 1)

    results = []
    # Skip index 0 (the book itself)
    for dist, idx in zip(distances.flatten()[1:], indices.flatten()[1:]):
        info = book_idx_to_info[idx]
        results.append({
            "Book-Title": info["Book-Title"],
            "Book-Author": info["Book-Author"],
            "Cosine-Similarity": round(float(1 - dist), 4)
        })

    print(f"Top {n} books similar to: '{matched_title}'")
    return pd.DataFrame(results)

# Quick test of Step 5.3
get_similar_books("the da vinci code", n=10)

Loaded book-user matrix: (5438, 6318) (rows=books, cols=users)
Top 10 books similar to: 'the da vinci code'


,Book-Title,Book-Author,Cosine-Similarity
0,angels &amp; demons,dan brown,0.2085
1,middlesex: a novel,jeffrey eugenides,0.1372
2,the lovely bones: a novel,alice sebold,0.1364
3,street dreams,faye kellerman,0.1296
4,widow's walk,robert b. parker,0.1267
5,the face,dean koontz,0.1257
6,digital fortress : a thriller,dan brown,0.1243
7,no second chance,harlan coben,0.1208
8,"the price of loyalty: george w. bush, the whit...",ron suskind,0.1159
9,touching evil,kay hooper,0.1158


In [26]:
# Precompute how many ratings each book actually has in the Gold matrix
book_gold_counts = np.asarray((book_user_matrix > 0).sum(axis=1)).flatten()

def recommend(user_id, n=10, high_rating_threshold=7, neighbors_per_book=15, min_book_ratings=5):
    if user_id not in user_id_to_idx:
        print(f"User {user_id} not found in Gold layer.")
        return pd.DataFrame()

    u_idx = user_id_to_idx[user_id]

    user_row = user_book_matrix[u_idx]
    rated_book_indices = user_row.indices
    rated_scores = user_row.data

    # Exclude both already-read ISBNs and already-read book titles (different editions)
    already_read_idx = set(rated_book_indices)
    already_read_titles = {book_idx_to_info[idx]["Book-Title"] for idx in rated_book_indices}

    high_rated = [
        (b_idx, rating)
        for b_idx, rating in zip(rated_book_indices, rated_scores)
        if rating >= high_rating_threshold
    ]
    if not high_rated:
        high_rated = sorted(zip(rated_book_indices, rated_scores), key=lambda x: x[1], reverse=True)[:5]

    candidate_scores = {}
    for b_idx, user_rating in high_rated:
        distances, indices = nn_model.kneighbors(
            book_user_matrix[b_idx],
            n_neighbors=neighbors_per_book + 1
        )
        for dist, neighbor_idx in zip(distances.flatten()[1:], indices.flatten()[1:]):
            if neighbor_idx in already_read_idx:
                continue
            if book_gold_counts[neighbor_idx] < min_book_ratings:
                continue
            title = book_idx_to_info[neighbor_idx]["Book-Title"]
            if title in already_read_titles:
                continue
            similarity = 1.0 - float(dist)
            if similarity > 0:
                candidate_scores[neighbor_idx] = candidate_scores.get(neighbor_idx, 0.0) + (similarity * float(user_rating))

    top_candidates = sorted(candidate_scores.items(), key=lambda x: x[1], reverse=True)

    recs = []
    seen_titles = set()
    for b_idx, score in top_candidates:
        info = book_idx_to_info[b_idx]
        if info["Book-Title"] in seen_titles:
            continue
        seen_titles.add(info["Book-Title"])
        recs.append({
            "Book-Title": info["Book-Title"],
            "Book-Author": info["Book-Author"],
            "Weighted-Score": round(score, 4)
        })
        if len(recs) == n:
            break

    return pd.DataFrame(recs)

print("Updated recommend(user_id, n=10) defined successfully.")

Updated recommend(user_id, n=10) defined successfully.


In [27]:
import pandas as pd

# Load user aggregates from Gold to pick 3 active users with varied tastes
user_agg = pd.read_parquet("gold/user_aggregates.parquet")
sample_users = (
    user_agg[user_agg["rating_count"].between(15, 30)]
    .sort_values("rating_count", ascending=False)
    .head(3)["User-ID"]
    .tolist()
)

def demo_user_recommendations(user_id, n=10, high_rating_threshold=7):
    u_idx = user_id_to_idx[user_id]
    user_row = user_book_matrix[u_idx]

    # Collect user's highly rated books
    liked_books = []
    for b_idx, rating in zip(user_row.indices, user_row.data):
        if rating >= high_rating_threshold:
            info = book_idx_to_info[b_idx]
            liked_books.append({
                "Liked Book": info["Book-Title"],
                "Author": info["Book-Author"],
                "User Rating": int(rating)
            })
    liked_df = (
        pd.DataFrame(liked_books)
        .sort_values("User Rating", ascending=False)
        .head(10)
        .reset_index(drop=True)
    )

    # Get recommendations
    recs_df = recommend(user_id, n=n, high_rating_threshold=high_rating_threshold).reset_index(drop=True)
    recs_df = recs_df.rename(columns={
        "Book-Title": "Recommended Book",
        "Book-Author": "Rec Author",
        "Weighted-Score": "Score"
    })

    # Combine side-by-side for easy visual comparison
    comparison = pd.concat([liked_df, recs_df], axis=1)

    print("=" * 90)
    print(f"USER ID: {user_id} | Showing Top {len(liked_df)} Highly Rated Books vs. Top {n} Recommendations")
    print("=" * 90)
    display(comparison)

for uid in sample_users:
    demo_user_recommendations(uid, n=10)

USER ID: 250709 | Showing Top 2 Highly Rated Books vs. Top 10 Recommendations


,Liked Book,Author,User Rating,Recommended Book,Rec Author,Score
0,the bridges of madison county,robert james waller,10.0,until the real thing comes along (ballantine r...,elizabeth berg,2.5516
1,slow waltz in cedar bend,robert james waller,10.0,iacocca: an autobiography,lee iacocca,2.5388
2,NaN,NaN,NaN,island of the sequined l,christopher moore,2.2362
3,NaN,NaN,NaN,lake news,barbara delinsky,2.1203
4,NaN,NaN,NaN,the doctor's book of home remedies : thousands...,editors prevention health books,2.0948
5,NaN,NaN,NaN,if there be thorns (dollanganger),v.c. andrews,1.8934
6,NaN,NaN,NaN,chicken soup for the mother's soul (chicken so...,jack canfield,1.8845
7,NaN,NaN,NaN,veronika decides to die,paulo coelho,1.5633
8,NaN,NaN,NaN,special delivery: a novel,danielle steel,1.5537
9,NaN,NaN,NaN,tell me your dreams,sidney sheldon,1.4097


USER ID: 276050 | Showing Top 10 Highly Rated Books vs. Top 10 Recommendations


,Liked Book,Author,User Rating,Recommended Book,Rec Author,Score
0,"the two towers (the lord of the rings, part 2)",j.r.r. tolkien,10,the hobbit : the enchanting prelude to the lor...,j.r.r. tolkien,8.8870
1,like water for chocolate: a novel in monthly i...,laura esquivel,10,the silmarillion,j.r.r. tolkien,5.4450
2,on the road,jack kerouac,10,crazy in alabama,mark childress,5.2858
3,"the return of the king (the lord of the rings,...",j.r.r. tolkien,10,shopaholic ties the knot,sophie kinsella,5.0877
4,the fellowship of the ring (the lord of the ri...,j.r.r. tolkien,10,lie down with lions,ken follett,4.1557
5,divine secrets of the ya-ya sisterhood : a novel,rebecca wells,9,the devil wears prada : a novel,lauren weisberger,4.0956
6,the nanny diaries: a novel,emma mclaughlin,9,l.a. confidential,james ellroy,3.9368
7,girl goddess #9 : nine stories,francesca lia block,9,b is for burglar (kinsey millhone mysteries (p...,sue grafton,3.7735
8,shopaholic takes manhattan (summer display opp...,sophie kinsella,9,prayers for the dead : a peter decker/rina laz...,faye kellerman,3.6732
9,the virgin suicides,jeffrey eugenides,9,j is for judgment,sue grafton,3.5839


USER ID: 271176 | Showing Top 10 Highly Rated Books vs. Top 10 Recommendations


,Liked Book,Author,User Rating,Recommended Book,Rec Author,Score
0,the hobbit : the enchanting prelude to the lor...,j.r.r. tolkien,10,"the two towers (the lord of the rings, part 2)",j.r.r. tolkien,13.0242
1,harry potter and the order of the phoenix (boo...,j. k. rowling,10,endymion,dan simmons,9.4257
2,harry potter and the chamber of secrets (book 2),j. k. rowling,10,harry potter and the sorcerer's stone (harry p...,j. k. rowling,7.9871
3,harry potter and the prisoner of azkaban (book 3),j. k. rowling,10,rise of endymion,dan simmons,7.5293
4,ender's game (ender wiggins saga (paperback)),orson scott card,10,xenocide (ender wiggins saga (paperback)),orson scott card,7.3180
5,harry potter and the goblet of fire (book 4),j. k. rowling,10,speaker for the dead (ender wiggins saga (pape...,orson scott card,6.2902
6,harry potter and the sorcerer's stone (book 1),j. k. rowling,10,the hobbit: or there and back again,j.r.r. tolkien,6.1401
7,the catcher in the rye,j.d. salinger,10,bridge of birds,barry hughart,5.9516
8,to say nothing of the dog,connie willis,10,children of the mind (ender wiggins saga (pape...,orson scott card,5.4892
9,doomsday book,connie willis,10,shadow of the hegemon (ender wiggins saga (pap...,orson scott card,4.9708
